# RLHF — full validated pipeline (beats 0.63)

Uses the **proven** recipe: **Qwen2.5-0.5B-Instruct** as both the RM backbone *and* the PPO policy (no weak SFT), **cleaned** UltraFeedback for the RM (`argilla/...-cleaned`). RM → PPO → eval. Reports RM accuracy (cleaned held-out **0.726** target + old H4 yardstick) **and** PPO-vs-Instruct win-rate. GPU-adaptive (T4 bf16 / P100 fp32). ~6-7 h on a T4.

In [ ]:
import torch, json
cap=torch.cuda.get_device_capability(0) if torch.cuda.is_available() else (0,0)
name=torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'NONE'
P100=tuple(cap)==(6,0); json.dump({'p100':P100},open('/tmp/gpu.json','w'))
print(f'GPU: {name} cap={cap} -> '+('P100 fp32' if P100 else 'T4 bf16'))

In [ ]:
!pip install -q 'transformers>=4.44,<5' 'datasets>=2.20' 'accelerate>=0.33' 'peft>=0.12' pytest
import json, subprocess, sys
if json.load(open('/tmp/gpu.json'))['p100']:
    subprocess.run([sys.executable,'-m','pip','install','-q','transformers>=4.44,<4.48'],check=True)
    subprocess.run([sys.executable,'-m','pip','install','-q','torch==2.4.1','torchvision==0.19.1',
                    'torchaudio==2.4.1','--index-url','https://download.pytorch.org/whl/cu121'],check=True)

In [ ]:
import os
!rm -rf /kaggle/working/rlhf-pipeline && git clone -q https://github.com/TheYellowDuck/RLHF-pipeline.git /kaggle/working/rlhf-pipeline
%cd /kaggle/working/rlhf-pipeline

In [ ]:
!python scripts/smoke_test.py 2>&1 | tail -2

## Config

In [ ]:
import json
P100=json.load(open('/tmp/gpu.json'))['p100']
MODEL='Qwen/Qwen2.5-0.5B-Instruct'
DATA_CLEAN='argilla/ultrafeedback-binarized-preferences-cleaned'
DATA_H4='HuggingFaceH4/ultrafeedback_binarized'
RM_SAMPLES=10000
DTYPE,BF16=('float32','false') if P100 else ('bfloat16','true')
print(f'model={MODEL} dtype={DTYPE} rm_samples={RM_SAMPLES}')

## 1. Reward model  (Instruct backbone + cleaned data → ~0.726)

In [ ]:
!python scripts/train_reward_model.py \
  -o model.name_or_path=$MODEL -o model.dtype=$DTYPE \
  -o data.name=$DATA_CLEAN -o data.train_split='train[2000:]' -o data.eval_split='train[:2000]' \
  -o data.max_samples=$RM_SAMPLES -o data.max_eval_samples=2000 -o data.max_length=512 \
  -o train.epochs=1 -o train.batch_size=4 -o train.grad_accum=4 -o train.bf16=$BF16 \
  -o train.lr=1.0e-5 -o train.label_smoothing=0.0 -o data.max_pair_similarity=1.0 \
  -o output_dir=checkpoints/reward_model

## 2. PPO  (Instruct policy, RL against the reward model)

In [ ]:
!python scripts/train_ppo.py \
  -o policy.name_or_path=$MODEL -o policy.dtype=$DTYPE \
  -o reward_model.name_or_path=checkpoints/reward_model -o reward_model.dtype=$DTYPE \
  -o data.name=$DATA_CLEAN -o data.train_split='train[2000:]' -o data.max_samples=6000 \
  -o ppo.total_episodes=2048 -o ppo.rollout_batch_size=16 -o ppo.mini_batch_size=2 \
  -o ppo.lr=1e-6 -o ppo.normalize_rewards=true \
  -o generation.max_new_tokens=40 -o data.max_prompt_length=160

## 3. Results → RESULTS.md

In [ ]:
import subprocess
def run(c):
    print('$',c,flush=True); o=subprocess.run(c,shell=True,capture_output=True,text=True)
    print(o.stdout[-700:]);
    if o.returncode: print(o.stderr[-1200:])
    return o.stdout
rc=run(f"python scripts/evaluate.py rm-accuracy --reward-model checkpoints/reward_model --data {DATA_CLEAN} --split 'train[:2000]' --max-samples 2000")
rh=run(f"python scripts/evaluate.py rm-accuracy --reward-model checkpoints/reward_model --data {DATA_H4} --split test_prefs --max-samples 2000")
wn=run(f"python scripts/evaluate.py score-policy --policy checkpoints/ppo --reward-model checkpoints/reward_model --compare {MODEL} --data {DATA_CLEAN} --split 'train[2000:]' --num 100 --max-new-tokens 40")
md=(f'# RLHF full pipeline results\n\n- backbone/policy: `{MODEL}`\n- RM data: `{DATA_CLEAN}` ({RM_SAMPLES} pairs)\n\n'
    f'## RM accuracy — CLEANED held-out (real number)\n```\n{rc}\n```\n\n'
    f'## RM accuracy — old H4 test (vs 0.63)\n```\n{rh}\n```\n\n'
    f'## PPO vs Instruct — win-rate + samples\n```\n{wn}\n```\n')
open('/kaggle/working/RESULTS.md','w').write(md); print('\nSaved -> RESULTS.md')

### Done
RM ~0.726 on cleaned labels; PPO win-rate > 50% means RL improved the policy vs the Instruct baseline.